In [1]:
print("Python kernel is working!")

Python kernel is working!


# For1Sight — Corrected ETL Pipeline

**Purpose:** Convert each driver's column-oriented `laptimes.json` arrays into one row per lap, apply lap-level validity filters, validate the result, and optionally write the Parquet warehouse.

## Why this is a separate notebook

- Keep `01_ETL_Pipeline(3).ipynb` unchanged as the original/rollback copy.
- This notebook preserves the source lap field names used by the For1Sight dashboard.
- The raw JSON files are read only. This notebook does not edit or delete them.
- Output writing is **disabled by default**. Review the validation results before enabling it.

**Important correction:** `del` and `iacc` are arrays in the raw JSON. They must be evaluated after exploding the arrays into individual lap records, not as whole-driver arrays.


## 1. Imports and Spark session

Run this notebook from the For1Sight project root, where the `raw_hf_data` folder is located.


In [5]:
import glob
import os
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    lit,
    input_file_name,
    regexp_replace,
    regexp_extract,
    arrays_zip,
    explode,
    size,
    array,
    array_min,
    array_max,
    array_repeat,
    coalesce,
    lower,
    upper,
    trim,
    when,
    count,
    sum as spark_sum,
)
from pyspark.sql.types import ArrayType

# Locate the project data folder without hard-coding one Windows username.
possible_raw_roots = [
    Path.cwd() / "raw_hf_data",
    Path.cwd().parent / "raw_hf_data",
]

RAW_ROOT = next((p.resolve() for p in possible_raw_roots if p.exists()), None)

if RAW_ROOT is None:
    raise FileNotFoundError(
        "Cannot find 'raw_hf_data'. Open this notebook from the For1Sight project "
        "root, or update possible_raw_roots to point to your raw_hf_data folder."
    )

PROJECT_ROOT = RAW_ROOT.parent
SPARK_TEMP = PROJECT_ROOT / "spark_temp"
SPARK_TEMP.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("For1Sight_Corrected_ETL")
    .config("spark.driver.memory", "4g")
    .config("spark.local.dir", str(SPARK_TEMP))
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark session ready.")
print("Project root:", PROJECT_ROOT)
print("Raw data root:", RAW_ROOT)


Spark session ready.
Project root: C:\Users\ayepd\Desktop\For1Sight Project
Raw data root: C:\Users\ayepd\Desktop\For1Sight Project\raw_hf_data


In [6]:
print("Spark Version:", spark.version)

print("Spark Application ID:", spark.sparkContext.applicationId)

print("Spark Master:", spark.sparkContext.master)

Spark Version: 3.5.3
Spark Application ID: local-1793572735262
Spark Master: local[*]


## 2. Discover and read the raw lap-time files

Each `laptimes.json` file contains arrays: each array index represents the same lap across the different fields. We first read each JSON file as one source record and retain its file path for metadata.


In [2]:
laptimes_files = sorted(
    glob.glob(str(RAW_ROOT / "**" / "laptimes.json"), recursive=True)
)

if not laptimes_files:
    raise FileNotFoundError(f"No laptimes.json files found under: {RAW_ROOT}")

print(f"Discovered {len(laptimes_files):,} laptimes.json files.")

lap_df = (
    spark.read
    .option("multiLine", "true")
    .json(laptimes_files)
    .withColumn("source_file", input_file_name())
)

raw_file_rows = lap_df.count()
print(f"Raw JSON records loaded: {raw_file_rows:,}")
lap_df.printSchema()


Discovered 6,904 laptimes.json files.
Raw JSON records loaded: 6,904
root
 |-- compound: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- dNum: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- del: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- delR: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- drv: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ff1G: array (nullable = true)
 |    |-- element: boolean (containsNull = true)
 |-- fresh: array (nullable = true)
 |    |-- element: boolean (containsNull = true)
 |-- iacc: array (nullable = true)
 |    |-- element: boolean (containsNull = true)
 |-- lSD: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- lST: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- lap: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- life:

## 3. Audit array lengths before flattening

`arrays_zip()` aligns values by array index. If arrays in one source file have different lengths, Spark pads shorter arrays with nulls. We therefore audit array lengths first and carry a mismatch flag into the flattened data.

Null optional arrays (a field absent from a particular file) are treated as empty arrays for the zip operation; they do not automatically count as a mismatch.


In [3]:
array_fields = [
    field for field in lap_df.schema.fields
    if isinstance(field.dataType, ArrayType)
]
array_cols = [field.name for field in array_fields]

if not array_cols:
    raise TypeError("No array columns were found in the laptimes JSON schema.")

# Ignore null/absent optional arrays when comparing lengths.
length_exprs = [
    when(col(field.name).isNotNull(), size(col(field.name)))
    for field in array_fields
]

array_length_audit = (
    lap_df
    .select(
        "source_file",
        array_min(array(*length_exprs)).alias("min_array_length"),
        array_max(array(*length_exprs)).alias("max_array_length"),
    )
    .withColumn(
        "array_length_mismatch",
        col("min_array_length").isNotNull()
        & col("max_array_length").isNotNull()
        & (col("min_array_length") != col("max_array_length"))
    )
)

mismatch_file_count = (
    array_length_audit
    .filter(col("array_length_mismatch"))
    .count()
)

print(f"Array columns discovered: {len(array_cols)}")
print(f"Files with mismatched non-null array lengths: {mismatch_file_count:,}")

array_length_audit.filter(
    col("array_length_mismatch")
).show(30, truncate=False)


Array columns discovered: 43
Files with mismatched non-null array lengths: 0
+-----------+----------------+----------------+---------------------+
|source_file|min_array_length|max_array_length|array_length_mismatch|
+-----------+----------------+----------------+---------------------+
+-----------+----------------+----------------+---------------------+



## 4. Explode the arrays into individual lap records

This is the key correction. The arrays are zipped together and exploded so that `del`, `iacc`, `lap`, `time`, sectors, tyre information and weather fields become values on the same individual lap row.

The original raw fields are retained. Additional metadata columns are derived from the source folder structure:

`raw_hf_data / year / event / session / driver / laptimes.json`


In [4]:
# Keep non-array fields (if any), the source path, and the array-length audit.
scalar_cols = [
    field.name for field in lap_df.schema.fields
    if not isinstance(field.dataType, ArrayType)
]

lap_with_audit = lap_df.join(
    array_length_audit,
    on="source_file",
    how="left"
)

# Replace null optional arrays with typed empty arrays before arrays_zip().
zip_inputs = []
for field in array_fields:
    empty_array = array_repeat(
        lit(None).cast(field.dataType.elementType),
        0
    )
    zip_inputs.append(
        coalesce(col(field.name), empty_array).alias(field.name)
    )

exploded = (
    lap_with_audit
    .select(
        *[col(name) for name in scalar_cols],
        col("min_array_length"),
        col("max_array_length"),
        col("array_length_mismatch"),
        explode(arrays_zip(*zip_inputs)).alias("_lap_record")
    )
)

flat_df = exploded.select(
    *[col(name) for name in scalar_cols],
    col("min_array_length"),
    col("max_array_length"),
    col("array_length_mismatch"),
    *[
        col("_lap_record").getField(field.name).alias(field.name)
        for field in array_fields
    ]
)

# Normalize Windows separators before extracting folder metadata.
flat_df = (
    flat_df
    .withColumn("_normalized_path", regexp_replace(col("source_file"), r"\\", "/"))
    .withColumn(
        "year",
        regexp_extract(
            col("_normalized_path"),
            r"(?i)(?:^|/)raw_hf_data/([^/]+)/",
            1
        ).cast("int")
    )
    .withColumn(
        "event_name",
        regexp_extract(
            col("_normalized_path"),
            r"(?i)(?:^|/)raw_hf_data/[^/]+/([^/]+)/",
            1
        )
    )
    .withColumn(
        "session_type",
        regexp_extract(
            col("_normalized_path"),
            r"(?i)(?:^|/)raw_hf_data/[^/]+/[^/]+/([^/]+)/",
            1
        )
    )
    .withColumn(
        "driver_folder",
        regexp_extract(
            col("_normalized_path"),
            r"(?i)(?:^|/)raw_hf_data/[^/]+/[^/]+/[^/]+/([^/]+)/laptimes\.json$",
            1
        )
    )
)

# Use the driver acronym in the JSON array; fall back to the driver folder if absent.
drv_text = lower(trim(col("drv").cast("string")))
driver_from_array = when(
    drv_text.isNull() | drv_text.isin("", "none", "null", "nan"),
    lit(None)
).otherwise(upper(trim(col("drv").cast("string"))))

flat_df = (
    flat_df
    .withColumn(
        "driver_code",
        coalesce(driver_from_array, upper(trim(col("driver_folder"))))
    )
    .withColumn("lap", col("lap").cast("int"))
    .withColumn("time", col("time").cast("double"))
)

print(f"Flattened lap-level rows: {flat_df.count():,}")
flat_df.select(
    "year", "event_name", "session_type", "driver_code",
    "lap", "time", "del", "iacc", "array_length_mismatch"
).show(10, truncate=False)


Flattened lap-level rows: 197,413
+----+------------------------+------------+-----------+---+-------+-----+-----+---------------------+
|year|event_name              |session_type|driver_code|lap|time   |del  |iacc |array_length_mismatch|
+----+------------------------+------------+-----------+---+-------+-----+-----+---------------------+
|2026|Pre-Season%20Testing%202|Practice%203|LIN        |1  |NULL   |false|false|false                |
|2026|Pre-Season%20Testing%202|Practice%203|LIN        |2  |97.386 |false|true |false                |
|2026|Pre-Season%20Testing%202|Practice%203|LIN        |3  |128.228|false|true |false                |
|2026|Pre-Season%20Testing%202|Practice%203|LIN        |4  |124.65 |false|true |false                |
|2026|Pre-Season%20Testing%202|Practice%203|LIN        |5  |96.609 |false|true |false                |
|2026|Pre-Season%20Testing%202|Practice%203|LIN        |6  |120.999|false|false|false                |
|2026|Pre-Season%20Testing%202|Practice

## 5. Clean individual lap rows

Correct interpretation of the source flags:

| Field | Meaning | Exclude when |
|---|---|---|
| `del` | Lap deleted | `true` |
| `iacc` | Lap timing is accurate | `false` |

Null flag values are retained as unknown rather than automatically rejected. Rows missing the essential lap number, lap time or driver code are excluded from the analytical warehouse.


In [5]:
# Remove only exact duplicate lap rows. Conflicting records with the same lap key
# are not silently collapsed; they are reported in the validation section.
flat_dedup_df = flat_df.dropDuplicates()

deleted_flag = coalesce(
    lower(trim(col("del").cast("string"))).isin("true", "1", "yes"),
    lit(False)
)

inaccurate_flag = coalesce(
    lower(trim(col("iacc").cast("string"))).isin("false", "0", "no"),
    lit(False)
)

# Keep the original flags and add explicit row-level diagnostic flags.
flagged_df = (
    flat_dedup_df
    .withColumn("is_deleted", deleted_flag)
    .withColumn("is_inaccurate", inaccurate_flag)
)

missing_critical = (
    col("lap").isNull()
    | col("time").isNull()
    | col("driver_code").isNull()
)

clean_df = flagged_df.filter(
    (~missing_critical)
    & (~col("is_deleted"))
    & (~col("is_inaccurate"))
)

# Dashboard compatibility: keep original columns such as time, lap, pos, drv,
# team, s1/s2/s3, tyre fields, plus year/event/session/driver_code metadata.
print("Flattened rows before exact deduplication:", f"{flat_df.count():,}")
print("Rows after exact deduplication:", f"{flat_dedup_df.count():,}")
print("Rows retained in clean warehouse:", f"{clean_df.count():,}")


Flattened rows before exact deduplication: 197,413
Rows after exact deduplication: 197,413
Rows retained in clean warehouse: 134,356


## 6. Data-quality validation

These checks do not write or overwrite any files. Review array mismatches, missing metadata and duplicate lap keys before publishing the warehouse.


In [6]:
# Counts by reason, measured on the deduplicated lap-level data.
deduplicated_count = flat_dedup_df.count()
deleted_count = flagged_df.filter(col("is_deleted")).count()
inaccurate_count = flagged_df.filter(col("is_inaccurate")).count()
missing_critical_count = flagged_df.filter(missing_critical).count()
clean_count = clean_df.count()

metadata_missing_count = clean_df.filter(
    col("year").isNull()
    | (col("event_name").isNull() | (trim(col("event_name")) == ""))
    | (col("session_type").isNull() | (trim(col("session_type")) == ""))
    | (col("driver_code").isNull() | (trim(col("driver_code")) == ""))
).count()

key_columns = ["year", "event_name", "session_type", "driver_code", "lap"]

duplicate_keys_df = (
    clean_df
    .groupBy(*key_columns)
    .count()
    .filter(col("count") > 1)
)

duplicate_key_groups = duplicate_keys_df.count()

print("=" * 72)
print("FOR1SIGHT CORRECTED ETL — VALIDATION SUMMARY")
print("=" * 72)
print(f"Source JSON files:                     {len(laptimes_files):,}")
print(f"Raw source records (one per file):     {raw_file_rows:,}")
print(f"Flattened lap rows:                    {flat_df.count():,}")
print(f"Rows after exact duplicate removal:    {deduplicated_count:,}")
print(f"Deleted lap rows excluded:             {deleted_count:,}")
print(f"Inaccurate lap rows excluded:          {inaccurate_count:,}")
print(f"Rows missing lap/time/driver code:     {missing_critical_count:,}")
print(f"Clean lap rows:                        {clean_count:,}")
print(f"Files with array length mismatch:      {mismatch_file_count:,}")
print(f"Clean rows with missing metadata:     {metadata_missing_count:,}")
print(f"Duplicate lap-key groups:              {duplicate_key_groups:,}")

print("\nDuplicate lap keys (if any):")
duplicate_keys_df.show(30, truncate=False)

print("\nClean warehouse schema:")
clean_df.printSchema()


FOR1SIGHT CORRECTED ETL — VALIDATION SUMMARY
Source JSON files:                     6,904
Raw source records (one per file):     6,904
Flattened lap rows:                    197,413
Rows after exact duplicate removal:    197,413
Deleted lap rows excluded:             1,974
Inaccurate lap rows excluded:          61,180
Rows missing lap/time/driver code:     28,261
Clean lap rows:                        134,356
Files with array length mismatch:      0
Clean rows with missing metadata:     0
Duplicate lap-key groups:              0

Duplicate lap keys (if any):
+----+----------+------------+-----------+---+-----+
|year|event_name|session_type|driver_code|lap|count|
+----+----------+------------+-----------+---+-----+
+----+----------+------------+-----------+---+-----+


Clean warehouse schema:
root
 |-- source_file: string (nullable = false)
 |-- min_array_length: integer (nullable = true)
 |-- max_array_length: integer (nullable = true)
 |-- array_length_mismatch: boolean (nullable = tr

In [9]:

from pyspark.sql.functions import (
    col, when, count, sum as F_sum, desc,
    regexp_replace
)

# ---------------------------------------------------------
# STEP 3.1 — INVESTIGATE FILTERING RESULTS
# ---------------------------------------------------------

diagnostic_df = (
    flagged_df
    .withColumn(
        "missing_critical",
        col("lap").isNull()
        | col("time").isNull()
        | col("driver_code").isNull()
    )
)

# Show combinations of exclusion reasons.
# This helps us understand overlapping exclusions.

reason_summary = (
    diagnostic_df
    .groupBy(
        "is_deleted",
        "is_inaccurate",
        "missing_critical"
    )
    .agg(count("*").alias("total_rows"))
    .orderBy(desc("total_rows"))
)

print("=" * 70)
print("EXCLUSION REASON COMBINATIONS")
print("=" * 70)

reason_summary.show(20, truncate=False)


# ---------------------------------------------------------
# STEP 3.2 — CLEAN RECORDS BY SEASON AND SESSION
# ---------------------------------------------------------

season_summary = (
    diagnostic_df
    .groupBy("year", "session_type")
    .agg(
        count("*").alias("original_rows"),

        F_sum(
            when(
                ~col("is_deleted")
                & ~col("is_inaccurate")
                & ~col("missing_critical"),
                1
            ).otherwise(0)
        ).alias("clean_rows"),

        F_sum(
            when(col("is_deleted"), 1).otherwise(0)
        ).alias("deleted_rows"),

        F_sum(
            when(col("is_inaccurate"), 1).otherwise(0)
        ).alias("inaccurate_rows")
    )
    .orderBy("year", "session_type")
)

print("=" * 70)
print("DATA DISTRIBUTION BY SEASON AND SESSION")
print("=" * 70)

season_summary.show(100, truncate=False)


# ---------------------------------------------------------
# STEP 3.3 — VERIFY EVENT NAME FORMATTING
# ---------------------------------------------------------

print("=" * 70)
print("EVENT NAME INSPECTION")
print("=" * 70)

(
    clean_df
    .select("year", "event_name")
    .distinct()
    .orderBy("year", "event_name")
    .show(100, truncate=False)
)


# ---------------------------------------------------------
# STEP 3.4 — VERIFY DRIVER COVERAGE
# ---------------------------------------------------------

print("=" * 70)
print("DRIVER COVERAGE BY SEASON")
print("=" * 70)

(
    clean_df
    .groupBy("year")
    .agg(
        count("*").alias("total_lap_records"),
        F_sum(
            when(col("driver_code").isNotNull(), 1)
            .otherwise(0)
        ).alias("records_with_driver_code")
    )
    .orderBy("year")
    .show(truncate=False)
)

print("STEP 3 VALIDATION COMPLETED")


EXCLUSION REASON COMBINATIONS
+----------+-------------+----------------+----------+
|is_deleted|is_inaccurate|missing_critical|total_rows|
+----------+-------------+----------------+----------+
|false     |false        |false           |134356    |
|false     |true         |false           |32822     |
|false     |true         |true            |28254     |
|true      |false        |false           |1870      |
|true      |true         |false           |104       |
|false     |false        |true            |7         |
+----------+-------------+----------------+----------+

DATA DISTRIBUTION BY SEASON AND SESSION
+----+-------------------+-------------+----------+------------+---------------+
|year|session_type       |original_rows|clean_rows|deleted_rows|inaccurate_rows|
+----+-------------------+-------------+----------+------------+---------------+
|2024|Practice%201       |12149        |7598      |0           |4551           |
|2024|Practice%202       |10091        |6785      |0   

In [1]:

from pyspark.sql.functions import (
    col,
    regexp_replace,
    upper,
    trim
)

# ==========================================================
# FOR1SIGHT STEP 4 — FINAL WAREHOUSE PREPARATION
# LIGHTWEIGHT VERSION
# ==========================================================

print("=" * 70)
print("FOR1SIGHT STEP 4 — WAREHOUSE PREPARATION")
print("=" * 70)

# 1. Start from validated clean data
warehouse_df = clean_df

# 2. Decode event and session names
warehouse_df = (
    warehouse_df
    .withColumn(
        "event_name",
        trim(regexp_replace(col("event_name"), "%20", " "))
    )
    .withColumn(
        "session_type",
        trim(regexp_replace(col("session_type"), "%20", " "))
    )
)

# 3. Standardise driver codes
warehouse_df = warehouse_df.withColumn(
    "driver_code",
    upper(trim(col("driver_code")))
)

# 4. Convert sector times into numeric values
for sector in ["s1", "s2", "s3"]:
    warehouse_df = warehouse_df.withColumn(
        sector,
        col(sector).cast("double")
    )

# 5. Remove temporary ETL columns
helper_columns = [
    "_normalized_path",
    "driver_folder",
    "min_array_length",
    "max_array_length",
    "array_length_mismatch",
    "is_deleted",
    "is_inaccurate"
]

warehouse_df = warehouse_df.drop(
    *[c for c in helper_columns
      if c in warehouse_df.columns]
)

# 6. Display schema only (no full dataset scan)
print("\nFINAL WAREHOUSE SCHEMA")
warehouse_df.printSchema()

# 7. Verify sector data types
print("\nSECTOR DATA TYPES")

for sector in ["s1", "s2", "s3"]:
    print(
        sector,
        warehouse_df.schema[sector].dataType
    )

# 8. Display column count
print("\nTotal columns:", len(warehouse_df.columns))

print("\n" + "=" * 70)
print("STEP 4 COMPLETED")
print("=" * 70)

print("No Parquet or HDFS files have been written.")


FOR1SIGHT STEP 4 — WAREHOUSE PREPARATION


NameError: name 'clean_df' is not defined

## 7. Regression check — SAI, Singapore Grand Prix, Race, Lap 26

This check is included to verify that a valid individual lap is retained after cleaning. It is a diagnostic check, not a substitute for the broader validation summary.


In [7]:
lap26_check = (
    clean_df
    .filter(
        (col("year") == 2026)
        & lower(col("event_name")).contains("singapore")
        & (lower(trim(col("session_type"))) == "race")
        & (upper(trim(col("driver_code"))) == "SAI")
        & (col("lap") == 26)
    )
    .withColumn(
        "sector_sum",
        col("s1").cast("double")
        + col("s2").cast("double")
        + col("s3").cast("double")
    )
    .withColumn(
        "sector_time_difference",
        col("time").cast("double") - col("sector_sum")
    )
)

lap26_count = lap26_check.count()
print(f"Matching clean SAI Singapore Race Lap 26 rows: {lap26_count}")

lap26_check.select(
    "year", "event_name", "session_type", "driver_code",
    "lap", "time", "s1", "s2", "s3",
    "sector_sum", "sector_time_difference", "del", "iacc"
).show(truncate=False)

if lap26_count == 0:
    print(
        "NOTE: The regression row was not found. Check the raw-data coverage "
        "and the extracted folder metadata before publishing."
    )


Matching clean SAI Singapore Race Lap 26 rows: 1
+----+------------------------+------------+-----------+---+-------+------+------+------+----------+----------------------+-----+----+
|year|event_name              |session_type|driver_code|lap|time   |s1    |s2    |s3    |sector_sum|sector_time_difference|del  |iacc|
+----+------------------------+------------+-----------+---+-------+------+------+------+----------+----------------------+-----+----+
|2026|Singapore%20Grand%20Prix|Race        |SAI        |26 |100.106|29.676|42.413|28.017|100.106   |0.0                   |false|true|
+----+------------------------+------------+-----------+---+-------+------+------+------+----------+----------------------+-----+----+



## 8. Optional warehouse export — disabled by default

**Do not enable writing until the validation summary has been reviewed.**

The output schema retains the original lap-time fields and adds metadata expected by the For1Sight dashboard (`year`, `event_name`, `session_type`, `driver_code`). The target paths match the existing project convention.

If there are array-length mismatches or missing metadata, the export is blocked by default. Investigate those files first rather than silently publishing padded or unidentified records.


In [8]:
# SAFETY SWITCH: leave False for the first run.
WRITE_OUTPUTS = False

# Only change these to True after reviewing all validation results.
ALLOW_ARRAY_MISMATCH_EXPORT = False
ALLOW_MISSING_METADATA_EXPORT = False

LOCAL_OUTPUT_PATH = PROJECT_ROOT / "for1sight_warehouse" / "hf_laptimes.parquet"
HDFS_OUTPUT_PATH = (
    "hdfs://localhost:9000/user/for1sight/warehouse/hf_laptimes.parquet"
)

if not WRITE_OUTPUTS:
    print("DRY RUN: no Parquet files were written or overwritten.")
else:
    if mismatch_file_count > 0 and not ALLOW_ARRAY_MISMATCH_EXPORT:
        raise RuntimeError(
            "Export blocked: array-length mismatches exist. "
            "Review array_length_audit or explicitly approve after investigation."
        )

    if metadata_missing_count > 0 and not ALLOW_MISSING_METADATA_EXPORT:
        raise RuntimeError(
            "Export blocked: clean rows have missing year/event/session/driver metadata. "
            "Fix path parsing or explicitly approve after investigation."
        )

    LOCAL_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

    # Write independently so an unavailable HDFS service does not prevent
    # the local dashboard warehouse from being produced.
    try:
        clean_df.write.mode("overwrite").parquet(str(LOCAL_OUTPUT_PATH))
        print("LOCAL PARQUET WRITE: SUCCESS")
        print("Path:", LOCAL_OUTPUT_PATH)
    except Exception as exc:
        print("LOCAL PARQUET WRITE: FAILED")
        print(repr(exc))

    try:
        clean_df.write.mode("overwrite").parquet(HDFS_OUTPUT_PATH)
        print("HDFS PARQUET WRITE: SUCCESS")
        print("Path:", HDFS_OUTPUT_PATH)
    except Exception as exc:
        print("HDFS PARQUET WRITE: FAILED")
        print(repr(exc))


DRY RUN: no Parquet files were written or overwritten.


## 9. Finish

After a successful export, run the For1Sight dashboard and verify that season, event, session and driver filters populate correctly. Keep the original notebook and raw JSON files as rollback sources until the new warehouse has been checked.


In [ ]:
# Optional: stop Spark when all checks and exports are complete.
# Uncomment only when you no longer need to run additional Spark cells.
# spark.stop()
